# Import dữ liệu

[Tài liệu tham khảo Data Cleaning](https://www.kaggle.com/code/ikramshah512/amazon-products-data-cleaning-feature-engineering)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
!pip install kagglehub[pandas-datasets]

import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "amazon_products_sales_data_uncleaned.csv"

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "ikramshah512/amazon-products-sales-dataset-42k-items-2025",
    file_path
)


100%|██████████| 36.8M/36.8M [00:00<00:00, 81.4MB/s]


# **Thăm dò cấu trúc dữ liệu**

In [ ]:
df.head()

,title,rating,number_of_reviews,bought_in_last_month,current/discounted_price,price_on_variant,listed_price,is_best_seller,is_sponsored,is_couponed,buy_box_availability,delivery_details,sustainability_badges,image_url,product_url,collected_at
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,4.6 out of 5 stars,375,300+ bought in past month,89.68,basic variant price: 2.4GHz,$159.00,No Badge,Sponsored,Save 15% with coupon,Add to cart,"Delivery Mon, Sep 1",Carbon impact,https://m.media-amazon.com/images/I/71pAqiVEs3...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",4.3 out of 5 stars,"2,457",6K+ bought in past month,9.99,basic variant price: nan,$15.99,No Badge,Sponsored,No Coupon,Add to cart,"Delivery Fri, Aug 29",NaN,https://m.media-amazon.com/images/I/61nbF6aVIP...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...",4.6 out of 5 stars,"3,044",2K+ bought in past month,314.00,basic variant price: nan,$349.00,No Badge,Sponsored,No Coupon,Add to cart,"Delivery Mon, Sep 1",NaN,https://m.media-amazon.com/images/I/61h78MEXoj...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
3,"Apple AirPods Pro 2 Wireless Earbuds, Active N...",4.6 out of 5 stars,"35,882",10K+ bought in past month,NaN,basic variant price: $162.24,No Discount,Best Seller,Organic,No Coupon,NaN,NaN,NaN,https://m.media-amazon.com/images/I/61SUj2aKoE...,/Apple-Cancellation-Transparency-Personalized-...,2025-08-21 11:14:29
4,Apple AirTag 4 Pack. Keep Track of and find Yo...,4.8 out of 5 stars,"28,988",10K+ bought in past month,NaN,basic variant price: $72.74,No Discount,No Badge,Organic,No Coupon,NaN,NaN,NaN,https://m.media-amazon.com/images/I/61bMNCeAUA...,/Apple-MX542LL-A-AirTag-Pack/dp/B0D54JZTHY/ref...,2025-08-21 11:14:29


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42675 entries, 0 to 42674
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   title                     42675 non-null  object
 1   rating                    41651 non-null  object
 2   number_of_reviews         41651 non-null  object
 3   bought_in_last_month      39458 non-null  object
 4   current/discounted_price  30926 non-null  object
 5   price_on_variant          42675 non-null  object
 6   listed_price              42675 non-null  object
 7   is_best_seller            42675 non-null  object
 8   is_sponsored              42675 non-null  object
 9   is_couponed               42675 non-null  object
 10  buy_box_availability      28022 non-null  object
 11  delivery_details          30955 non-null  object
 12  sustainability_badges     3408 non-null   object
 13  image_url                 42675 non-null  object
 14  product_url           

In [ ]:
df.apply(lambda col: col.astype(str).str.strip().isin([
    "", "N/A", "NA", "na", "n/a",
    "None", "none", "NULL", "null",
    "-", "--", "unknown", "Unknown"
]).sum())


,0
title,0
rating,0
number_of_reviews,0
bought_in_last_month,0
current/discounted_price,0
price_on_variant,0
listed_price,0
is_best_seller,0
is_sponsored,0
is_couponed,0


In [ ]:
df.isnull().sum()

,0
title,0
rating,1024
number_of_reviews,1024
bought_in_last_month,3217
current/discounted_price,11749
price_on_variant,0
listed_price,0
is_best_seller,0
is_sponsored,0
is_couponed,0


In [ ]:
df1 = df.copy()

num_duplicates = df1.duplicated().sum()
print(f"Số hàng bị duplicates: {num_duplicates}")

Số hàng bị duplicates: 0


# Các vấn đề của bộ dữ liệu 1
1.   Sai định dạng:
  - rating: nội dung không cần thiết "out of 5 stars"
  - number_of_reviews: dấu ","
  - bought_in_last_month: nội dung, ký hiệu không cần thiết "bought in past month", "K"
  - current/discounted_price: có dấu ","
  - price_on_variant: Cột này chứa từ ngữ không cần thiết, nhiều giá trị có thông tin sai thay vì "giá", chứa các ký tự đặc biệt.
  - listed_price: chứa ký tự đặc biệt, dữ liệu không đồng nhất, cần chuẩn hóa.
  - is_couponed và buy_box_availability: cần chuẩn hóa
  - delivery_details: định dạng không nhất quán
  - product_url: Cần bổ sung thêm tên miền chính của trang web.
  2. Sai kiểu dữ liệu:
  - numeric: rating, number_of_reviews, bought_in_last_month,current/discounted_price, price_on_variant, listed_price.
  - date: delivery_details, collected_at
3. Giá trị khuyết:
  - Các cột có giá trị khuyết: rating, number_of_reviews, bought_in_last_month, current/discounted_price, buy_box_availability, delivery_details, sustainability_badges, product_url
  

# Làm sạch cột Rating

In [ ]:
df1['rating']

,rating
0,4.6 out of 5 stars
1,4.3 out of 5 stars
2,4.6 out of 5 stars
3,4.6 out of 5 stars
4,4.8 out of 5 stars
...,...
42670,5.0 out of 5 stars
42671,4.2 out of 5 stars
42672,4.3 out of 5 stars
42673,4.7 out of 5 stars


In [ ]:
df1['rating'].isnull().sum()

np.int64(1024)

In [ ]:
df1['rating'] = df1['rating'].str.replace(r'out of 5 stars', '').str.strip().astype(float)

In [ ]:
rating_counts = df1['rating'].value_counts(dropna=False)
print(rating_counts)

rating
4.6    6151
4.4    5525
4.5    5359
4.7    4664
4.8    4230
4.3    2927
4.2    2837
4.1    1959
4.0    1465
3.9    1316
3.8    1083
NaN    1024
5.0     995
4.9     704
3.6     666
3.7     604
3.2     363
3.5     242
3.4     216
3.0     148
2.0     143
2.7      15
1.5      15
3.3       6
2.8       4
1.0       4
2.4       3
2.9       2
3.1       2
2.3       1
2.5       1
2.6       1
Name: count, dtype: int64


# Làm sạch cột number_of_reviews

In [ ]:
df1['number_of_reviews']

,number_of_reviews
0,375
1,"2,457"
2,"3,044"
3,"35,882"
4,"28,988"
...,...
42670,1
42671,20
42672,57
42673,"7,102"


In [ ]:
df1['number_of_reviews'].isnull().sum()

np.int64(1024)

In [ ]:
df1['number_of_reviews'] = df1['number_of_reviews'].str.replace(',','').str.strip().astype(float)

In [ ]:
reviews_counts = df1['number_of_reviews'].value_counts(dropna=False)
print(reviews_counts)

number_of_reviews
NaN         1024
25.0         626
1.0          524
8.0          478
30.0         409
            ... 
3986.0         1
186977.0       1
4629.0         1
3650.0         1
5959.0         1
Name: count, Length: 4414, dtype: int64


# Làm sạch cột bought_in_last_month

In [ ]:
df1['bought_in_last_month']

,bought_in_last_month
0,300+ bought in past month
1,6K+ bought in past month
2,2K+ bought in past month
3,10K+ bought in past month
4,10K+ bought in past month
...,...
42670,100+ bought in past month
42671,200+ bought in past month
42672,50+ bought in past month
42673,500+ bought in past month


In [ ]:
df1['bought_in_last_month'].isnull().sum()

np.int64(3217)

In [ ]:
df1['bought_in_last_month'] = df1['bought_in_last_month'].str.replace('+ bought in past month', '').str.strip().str.replace('K', '000')
df1['bought_in_last_month'] = df1['bought_in_last_month'].where(df1['bought_in_last_month'].str.isdigit(), np.nan)
df1['bought_in_last_month'] = (df1['bought_in_last_month'].where(df1['bought_in_last_month'].str.isdigit(), np.nan).astype('Int64'))

In [ ]:
bought_counts = df1['bought_in_last_month'].value_counts()
print(bought_counts)

bought_in_last_month
100       8801
50        5967
200       5645
300       2842
500       2351
1000      2084
400       1436
20000      772
2000       426
800        280
3000       249
10000      229
600        229
4000       196
5000       120
700        102
90000       92
6000        81
900         63
7000        49
9000        44
8000        36
30000       28
40000       14
100000      13
50000        6
70000        4
60000        3
80000        2
Name: count, dtype: Int64


In [ ]:
df1['bought_in_last_month'].isnull().sum()

np.int64(10511)

# Làm sạch cột price_on_variant

In [ ]:
df1['price_on_variant'] = df['price_on_variant']

In [ ]:
df1['price_on_variant'].isnull().sum()

np.int64(0)

In [ ]:
df1['price_on_variant']

,price_on_variant
0,basic variant price: 2.4GHz
1,basic variant price: nan
2,basic variant price: nan
3,basic variant price: $162.24
4,basic variant price: $72.74
...,...
42670,basic variant price: nan
42671,basic variant price: $25.00 off coupon applied
42672,basic variant price: Lowest price in 30 days
42673,basic variant price: nan


In [ ]:
#Tách giá bằng chuỗi bằng cách chỉ lấy giá trị sau ":"
df1['price_on_variant'] = df1['price_on_variant'].str.split(':').str[1]
df1['price_on_variant'].head()

,price_on_variant
0,2.4GHz
1,nan
2,nan
3,$162.24
4,$72.74


In [ ]:
df1['price_on_variant'] = df1['price_on_variant'].str.strip()
#Chỉ giữ những dòng có ký hiệu "$", còn lại chuyển sang NaN
df1.loc[
    ~df1['price_on_variant'].str.contains(r'\$', regex=True),
    'price_on_variant'
] = np.nan

df1['price_on_variant'].head()

,price_on_variant
0,NaN
1,NaN
2,NaN
3,$162.24
4,$72.74


In [ ]:
df1['price_on_variant'] = df1['price_on_variant'].str.split(' ').str[0]

df1['price_on_variant'].head()

,price_on_variant
0,NaN
1,NaN
2,NaN
3,$162.24
4,$72.74


In [ ]:
df1['price_on_variant'] = (
    df1['price_on_variant']
    .str.replace(r"\$", "", regex=True)
    .str.replace(",", "")
    .astype(float)
)
df1['price_on_variant'].head()

,price_on_variant
0,NaN
1,NaN
2,NaN
3,162.24
4,72.74


# Làm sạch cột current/discounted_price

In [ ]:
df1['current/discounted_price'] = df['current/discounted_price']

In [ ]:
df1['current/discounted_price'].isnull().sum()

np.int64(11749)

In [ ]:
df1['current/discounted_price'].head()

,current/discounted_price
0,89.68
1,9.99
2,314.00
3,NaN
4,NaN


In [ ]:
# Xóa dấu "," và đổi kiểu dữ liệu
df1['current/discounted_price'] = df1['current/discounted_price'].str.replace(',', '').astype(float)
df1['current/discounted_price'].head()

,current/discounted_price
0,89.68
1,9.99
2,314.00
3,NaN
4,NaN


# Làm sạch cột listed_price

In [ ]:
df1['listed_price'] = df['listed_price']

In [ ]:
listed_price_counts = df1['listed_price'].value_counts()
print(listed_price_counts)

listed_price
No Discount    30364
$79.99           500
$29.99           406
$16.61           360
$23.26           283
               ...  
$42.50             1
$2,999.99          1
$16.97             1
$71.87             1
$14.33             1
Name: count, Length: 911, dtype: int64


In [ ]:
df1['listed_price'] = df1['listed_price'].str.replace('$','').str.strip().str.replace(',','').str.strip()
df1['listed_price'].head()

,listed_price
0,159.00
1,15.99
2,349.00
3,No Discount
4,No Discount


In [ ]:
df1['listed_price'] = df1['listed_price'].astype(str).str.strip()
df1.loc[
    df1['listed_price'] == 'No Discount',
    'listed_price'
] = df1['current/discounted_price']

In [ ]:
df1['listed_price'] = df1['listed_price'].astype(float)

In [ ]:
df1[['listed_price','current/discounted_price']].sample(5)

,listed_price,current/discounted_price
1030,NaN,NaN
33821,NaN,NaN
38489,NaN,NaN
5812,119.99,119.99
13101,43.99,29.99


In [ ]:
df1['listed_price'].isnull().sum()

np.int64(11749)

# Làm sạch cột delivery_details

In [ ]:
df1['delivery_details']

,delivery_details
0,"Delivery Mon, Sep 1"
1,"Delivery Fri, Aug 29"
2,"Delivery Mon, Sep 1"
3,NaN
4,NaN
...,...
42670,"FREE delivery Thu, Sep 4Or fastest delivery Tu..."
42671,"FREE delivery Thu, Sep 4Or fastest delivery Mo..."
42672,"FREE delivery Thu, Sep 4Or fastest delivery We..."
42673,"FREE delivery Thu, Sep 4 on $35 of items shipp..."


In [ ]:
unique_delivery_counts = df1['delivery_details'].value_counts().unique()
print(unique_delivery_counts)

[6189 4364 3700 3278 2164 1238 1012 1003  771  578  545  501  460  413
  357  319  253  192  185  150  129  111  109   98   94   93   92   90
   85   83   80   74   73   68   67   63   62   60   57   55   54   48
   46   40   39   38   37   36   34   33   32   30   29   27   24   22
   21   20   19   16   15   14   13   12   11   10    9    8    7    6
    5    4    3    2    1]


In [ ]:
df1['delivery_details'] = df1['delivery_details'].str.extract(r'(?:Mon|Tue|Wed|Thu|Fri|Sat|Sun)?,?\s*(\w+\s+\d{1,2})')

In [ ]:
df1['delivery_details'] = pd.to_datetime(df1['delivery_details'] + ' 2025', errors='coerce')

In [ ]:
df1['delivery_details']

,delivery_details
0,2025-09-01
1,2025-08-29
2,2025-09-01
3,NaT
4,NaT
...,...
42670,2025-09-04
42671,2025-09-04
42672,2025-09-04
42673,2025-09-04


# Làm sạch cột Product_URL

In [ ]:
#Xem giá trị của cột
df1['product_url']

,product_url
0,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...
1,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...
2,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...
3,/Apple-Cancellation-Transparency-Personalized-...
4,/Apple-MX542LL-A-AirTag-Pack/dp/B0D54JZTHY/ref...
...,...
42670,/Elgato-4K-Pro-Internal-Capture/dp/B0DLR3WQWR/...
42671,/Arlo-Essential-Spotlight-Camera-Surveillance/...
42672,/GIGABYTE-FO32U2-32-3840x2160-240Hz-FreeSync-A...
42673,/Monoprice-XLR-Male-4-Inch-Cable/dp/B001UJEKZ6...


In [ ]:
# Kiểm tra chéo mục nhập đầu tiên của cột 'product_url' và 'image_url'
first_product_url = df1['product_url'].iloc[0]
first_image_url = df1['image_url'].iloc[0]

print(f"First product URL: https://www.amazon.com{first_product_url}")
print(f"First image URL: {first_image_url}")

First product URL: https://www.amazon.com/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxNDQ2OjE3NTU4MDAwNjg6c3BfYXRmX2Jyb3dzZTozMDA2NzE0NTMwMTcyMDI6OjA6Og&url=%2FBOYA-BOYALINK-Microphone-Micophone-Cancelling%2Fdp%2FB0DNZB7TQG%2Fref%3Dsr_1_1_sspa%3Fdib%3DeyJ2IjoiMSJ9.avmZlHCQuVOwikquBqYSIjN8SVcyxtkXHQMPt7Zjzkf4TeZzrZfQETMhdWuWgtTrVz8ITKpXLHvZj0fZRjxqgMPYNMitRqeUoeIwdYfc5nnzJ8m0T8HYeedlh3YSOhJQjeHskevMUQWyg6TtoB2tHcHt-edYPsQ6VwFQTI6avsPrgVpFKrto3ff9TDR9BcRyPwM6AiYn-vh7wA5PP9DjZddhCPf7bVPcHMZ6Hwd40dQDWm9_M8R-LcwKY8wnuWRUSplhfYJBvTjAgsb3Y3y88VGpfqY3V8Rd2ge-woBzUMA.yqwi-krmElGgXIMa8kKUPx1XmaXd9lHaAkMkVVpStxo%26dib_tag%3Dse%26qid%3D1755800068%26refinements%3Dp_n_g-101014971069111%253A119653281011%26s%3Delectronics%26sr%3D1-1-spons%26sp_csd%3Dd2lkZ2V0TmFtZT1zcF9hdGZfYnJvd3Nl%26psc%3D1
First image URL: https://m.media-amazon.com/images/I/71pAqiVEs3L._AC_UL320_.jpg


In [ ]:
df1['product_url'].isna().sum()

np.int64(2069)

In [ ]:
amazon_base_url = "https://www.amazon.com"
df1['product_url'] = df1['product_url'].apply(
    lambda x: amazon_base_url + x
    if pd.notna(x) and not str(x).startswith(("http://", "https://"))
    else x
)

In [ ]:
df1['product_url'].isna().sum()

np.int64(2069)

In [ ]:
df1['product_url']

,product_url
0,https://www.amazon.com/sspa/click?ie=UTF8&spc=...
1,https://www.amazon.com/sspa/click?ie=UTF8&spc=...
2,https://www.amazon.com/sspa/click?ie=UTF8&spc=...
3,https://www.amazon.com/Apple-Cancellation-Tran...
4,https://www.amazon.com/Apple-MX542LL-A-AirTag-...
...,...
42670,https://www.amazon.com/Elgato-4K-Pro-Internal-...
42671,https://www.amazon.com/Arlo-Essential-Spotligh...
42672,https://www.amazon.com/GIGABYTE-FO32U2-32-3840...
42673,https://www.amazon.com/Monoprice-XLR-Male-4-In...


# Làm sạch cột Collected_At

In [ ]:
df1['collected_at']

,collected_at
0,2025-08-21 11:14:29
1,2025-08-21 11:14:29
2,2025-08-21 11:14:29
3,2025-08-21 11:14:29
4,2025-08-21 11:14:29
...,...
42670,2025-08-30 19:56:33
42671,2025-08-30 19:56:33
42672,2025-08-30 19:56:33
42673,2025-08-30 19:56:33


In [ ]:
df1['collected_at'] = pd.to_datetime(df1['collected_at'])

In [ ]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42675 entries, 0 to 42674
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   title                     42675 non-null  object        
 1   rating                    41651 non-null  float64       
 2   number_of_reviews         41651 non-null  float64       
 3   bought_in_last_month      32164 non-null  Int64         
 4   current/discounted_price  30926 non-null  float64       
 5   price_on_variant          20071 non-null  float64       
 6   listed_price              30926 non-null  float64       
 7   is_best_seller            42675 non-null  object        
 8   is_sponsored              42675 non-null  object        
 9   is_couponed               42675 non-null  object        
 10  buy_box_availability      28022 non-null  object        
 11  delivery_details          30692 non-null  datetime64[ns]
 12  sustainability_bad

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df1.to_pickle('/content/drive/MyDrive/Colab Notebooks/DS111/cleaned_data.pkl')